# ONLY Fourier inference timing: W1 vs Panel A alternatives

This Kaggle notebook measures latency for four already-trained checkpoints in one run.

The dataset is downloaded and split once. The test images are loaded once. W1 receives
the Fourier-transformed images; the other three methods receive the original images.


## Configuration

Upload all four `.pt` files to Kaggle and replace the four entries in `MODEL_PATHS`.
Then run the notebook once. Missing paths are reported and skipped.


In [ ]:
from pathlib import Path

EXECUTION_PLAN = 'only_fourier_inference_timing_kaggle_seed42'
SMOKE_RUN = False
SEED = 42

# Upload all four .pt files to Kaggle, then replace these four paths.
MODEL_PATHS = {
    'w1_fourier_highpass': '/kaggle/input/UPLOAD_W1_DATASET/best.pt',
    'n1_erasing': '/kaggle/input/UPLOAD_ERASING_DATASET/best.pt',
    'n1_flipud': '/kaggle/input/UPLOAD_FLIPUD_DATASET/best.pt',
    'n2_scale_erasing': '/kaggle/input/UPLOAD_SCALE_ERASING_DATASET/best.pt',
}

FOURIER_SIGMA = 50.0
FOURIER_ALPHA = 0.10
IMAGE_SIZE = 640
WARMUP_IMAGES = 10
FOURIER_BATCH_SIZE = 16
DEVICE = 0

if Path('/kaggle/working').exists():
    WORK_DIR = Path('/kaggle/working')
else:
    WORK_DIR = Path.cwd()

DATASET_DIR = WORK_DIR / 'shrimpDisHandSegV2-1'
EXPERIMENT_ROOT = WORK_DIR / EXECUTION_PLAN
REPORT_DIR = EXPERIMENT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Configured methods:', list(MODEL_PATHS))
print('WORK_DIR:', WORK_DIR)
print('IMAGE_SIZE:', IMAGE_SIZE)


## Install Dependencies

Run this cell once in a fresh Kaggle session.


In [ ]:
%pip install -q ultralytics==8.4.62 roboflow opencv-python-headless pyyaml

In [ ]:
import importlib.metadata
import ultralytics

print('Ultralytics:', importlib.metadata.version('ultralytics'))
print('Required:', '8.4.62')


## Download Roboflow Dataset

In [ ]:
import os
from roboflow import Roboflow

ROBOFLOW_API_KEY_DIRECT = 'KOEk0qLzBFDc7zfyxtgs'
ROBOFLOW_WORKSPACE = 'lets-try-this'
ROBOFLOW_PROJECT = 'shrimpdishandsegv2'
ROBOFLOW_VERSION = 1
ROBOFLOW_FORMAT = 'yolo26'

def get_roboflow_api_key():
    if ROBOFLOW_API_KEY_DIRECT.strip():
        return ROBOFLOW_API_KEY_DIRECT.strip()
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
    except Exception:
        return os.environ.get('ROBOFLOW_API_KEY', '').strip()

api_key = get_roboflow_api_key()
if not api_key:
    raise RuntimeError('Missing Roboflow API key. Configure ROBOFLOW_API_KEY in Kaggle Secrets.')

rf = Roboflow(api_key=api_key)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)
version.download(ROBOFLOW_FORMAT, location=str(DATASET_DIR))
base_path = str(DATASET_DIR)
data_yaml_path = str(DATASET_DIR / 'data.yaml')
print('Dataset path:', base_path)


## Leakage-Safe Grouped Split

In [ ]:
import os
import random
import re
import shutil
from collections import defaultdict, Counter
from pathlib import Path

SEED = 42
random.seed(SEED)

base_path = str(DATASET_DIR)
train_path = os.path.join(base_path, 'train')
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

# Prevent leakage from multiple photos of the same shrimp.
# Expected original filename: <diseasename>-<shrimpid>-img-<imgnum>.jpg
# Roboflow may export names like <diseasename>-<shrimpid>-img-<imgnum>_jpg.rf.<hash>.jpg.
# Example disease names: Healthy, BG, WSSV_BG, WSSV.
GROUP_SPLIT_BY_SHRIMP = True
GROUP_STRATIFY_BY_DISEASE = True
REBUILD_SPLIT_FROM_ALL_SPLITS = True
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10

SHRIMP_NAME_PATTERN = re.compile(
    r'^(?P<disease>Healthy|BG|WSSV_BG|WSSV)-(?P<shrimp_id>.+)-img-(?P<img_num>\d+)$',
    re.IGNORECASE,
)


def normalize_roboflow_stem(stem):
    """Recover the original filename stem from Roboflow-exported names."""
    stem = re.sub(r'_(jpg|jpeg|png|bmp|webp)\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)
    stem = re.sub(r'\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)
    return stem


for split in ['train', 'valid', 'test']:
    for sub in ['images', 'labels']:
        os.makedirs(os.path.join(base_path, split, sub), exist_ok=True)


def parse_shrimp_group_key(image_name):
    """Return a stable group key so all images from one shrimp stay in one split."""
    stem = normalize_roboflow_stem(Path(image_name).stem)
    match = SHRIMP_NAME_PATTERN.match(stem)
    if not match:
        return f'unparsed::{Path(image_name).stem}', 'unparsed', None, None

    disease = match.group('disease')
    shrimp_id = match.group('shrimp_id')
    img_num = int(match.group('img_num'))
    group_key = f'{disease.lower()}::{shrimp_id}'
    return group_key, disease, shrimp_id, img_num


def image_files_in_split(split):
    image_dir = Path(base_path) / split / 'images'
    return sorted(
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )


def move_image_and_label(image_path, target_split):
    target_img_dir = Path(base_path) / target_split / 'images'
    target_lbl_dir = Path(base_path) / target_split / 'labels'
    target_img_dir.mkdir(parents=True, exist_ok=True)
    target_lbl_dir.mkdir(parents=True, exist_ok=True)

    label_name = f'{image_path.stem}.txt'
    label_src = image_path.parent.parent / 'labels' / label_name
    image_dst = target_img_dir / image_path.name
    label_dst = target_lbl_dir / label_name

    if image_path.resolve() != image_dst.resolve():
        if image_dst.exists():
            raise FileExistsError(f'Duplicate image destination would be overwritten: {image_dst}')
        shutil.move(str(image_path), str(image_dst))

    if label_src.exists():
        if label_src.resolve() != label_dst.resolve():
            if label_dst.exists():
                raise FileExistsError(f'Duplicate label destination would be overwritten: {label_dst}')
            shutil.move(str(label_src), str(label_dst))
    else:
        label_dst.write_text('')


def rebuild_train_pool_from_all_splits():
    all_images = []
    for split in ['train', 'valid', 'test']:
        all_images.extend(image_files_in_split(split))

    for image_path in sorted(all_images):
        move_image_and_label(image_path, 'train')

    return image_files_in_split('train')


def remove_yolo_label_caches(root):
    for cache_path in Path(root).glob('**/*.cache'):
        cache_path.unlink()
        print(f'Removed stale cache: {cache_path}')


def disease_for_group(filenames):
    diseases = []
    for filename in filenames:
        _, disease, _, _ = parse_shrimp_group_key(filename)
        diseases.append(disease)
    counts = Counter(diseases)
    if len(counts) > 1:
        print(f'Warning: group has mixed disease names: {dict(counts)}')
    return counts.most_common(1)[0][0]


def split_one_stratum(items):
    n = len(items)
    train_count = int(TRAIN_RATIO * n)
    val_count = int(VAL_RATIO * n)
    test_count = n - train_count - val_count

    if n >= 3:
        if val_count == 0:
            val_count = 1
            train_count -= 1
        if test_count == 0:
            test_count = 1
            train_count -= 1
    if train_count < 1 and n > 0:
        train_count = 1
    while train_count + val_count + test_count > n:
        train_count -= 1
    test_count = n - train_count - val_count

    return (
        items[:train_count],
        items[train_count:train_count + val_count],
        items[train_count + val_count:],
    )


def grouped_stratified_split(group_items):
    strata = defaultdict(list)
    for group_key, filenames in group_items:
        strata[disease_for_group(filenames)].append((group_key, filenames))

    split_to_groups = {'train': [], 'valid': [], 'test': []}
    rng = random.Random(SEED)
    for disease, items in sorted(strata.items()):
        items = sorted(items, key=lambda item: item[0])
        rng.shuffle(items)
        train_items, val_items, test_items = split_one_stratum(items)
        split_to_groups['train'].extend(train_items)
        split_to_groups['valid'].extend(val_items)
        split_to_groups['test'].extend(test_items)
        print(
            f'  - {disease}: {len(train_items)} train groups, '
            f'{len(val_items)} valid groups, {len(test_items)} test groups'
        )

    for split in split_to_groups:
        split_to_groups[split] = sorted(split_to_groups[split], key=lambda item: item[0])
    return split_to_groups


def grouped_random_split(group_items):
    group_items = sorted(group_items, key=lambda item: item[0])
    random.Random(SEED).shuffle(group_items)
    n_groups = len(group_items)
    train_group_count = int(TRAIN_RATIO * n_groups)
    val_group_count = int(VAL_RATIO * n_groups)
    return {
        'train': group_items[:train_group_count],
        'valid': group_items[train_group_count:train_group_count + val_group_count],
        'test': group_items[train_group_count + val_group_count:],
    }


def split_summary(split_groups):
    group_diseases = Counter()
    image_diseases = Counter()
    for _, filenames in split_groups:
        group_diseases[disease_for_group(filenames)] += 1
        for filename in filenames:
            _, disease, _, _ = parse_shrimp_group_key(filename)
            image_diseases[disease] += 1
    return group_diseases, image_diseases


def split_grouped_by_shrimp():
    if REBUILD_SPLIT_FROM_ALL_SPLITS:
        image_paths = rebuild_train_pool_from_all_splits()
    else:
        image_paths = image_files_in_split('train')

    groups = defaultdict(list)
    disease_counts = Counter()
    unparsed = []

    for image_path in image_paths:
        group_key, disease, shrimp_id, img_num = parse_shrimp_group_key(image_path.name)
        groups[group_key].append(image_path.name)
        disease_counts[disease] += 1
        if disease == 'unparsed':
            unparsed.append(image_path.name)

    group_items = sorted(groups.items(), key=lambda item: item[0])
    if GROUP_STRATIFY_BY_DISEASE:
        print('Building shrimp-grouped, disease-stratified split:')
        split_to_groups = grouped_stratified_split(group_items)
    else:
        print('Building shrimp-grouped random split:')
        split_to_groups = grouped_random_split(group_items)

    for split, split_groups in split_to_groups.items():
        for _, filenames in split_groups:
            for filename in filenames:
                move_image_and_label(Path(base_path) / 'train' / 'images' / filename, split)

    print('Shrimp-grouped split complete:')
    for split, split_groups in split_to_groups.items():
        image_count = sum(len(filenames) for _, filenames in split_groups)
        group_diseases, image_diseases = split_summary(split_groups)
        print(f'  - {split}: {len(split_groups)} shrimp groups, {image_count} images')
        print(f'    group disease counts: {dict(sorted(group_diseases.items()))}')
        print(f'    image disease counts: {dict(sorted(image_diseases.items()))}')

    print('Source filename disease counts before split:', dict(sorted(disease_counts.items())))
    if unparsed:
        print(f'Warning: {len(unparsed)} filenames did not match the shrimp naming pattern. They were split as single-image groups.')
        print('First unparsed examples:', unparsed[:10])

    group_to_split = {}
    leakage = []
    for split in ['train', 'valid', 'test']:
        for image_path in image_files_in_split(split):
            group_key, *_ = parse_shrimp_group_key(image_path.name)
            previous_split = group_to_split.setdefault(group_key, split)
            if previous_split != split:
                leakage.append((group_key, previous_split, split, image_path.name))

    if leakage:
        raise RuntimeError(f'Shrimp-level split leakage detected: {leakage[:10]}')
    print('Shrimp-level leakage check passed.')
    remove_yolo_label_caches(base_path)


if GROUP_SPLIT_BY_SHRIMP:
    split_grouped_by_shrimp()
else:
    valid_images_dir = Path(base_path) / 'valid' / 'images'
    test_images_dir = Path(base_path) / 'test' / 'images'

    if not any(valid_images_dir.glob('*')) and not any(test_images_dir.glob('*')):
        image_files = sorted(
            f for f in os.listdir(os.path.join(train_path, 'images'))
            if f.lower().endswith(IMAGE_EXTENSIONS)
        )
        random.shuffle(image_files)

        train_count = int(0.8 * len(image_files))
        val_count = int(0.1 * len(image_files))
        val_files = image_files[train_count:train_count + val_count]
        test_files = image_files[train_count + val_count:]

        def move_files(files, target_split):
            for f in files:
                move_image_and_label(Path(train_path) / 'images' / f, target_split)

        move_files(val_files, 'valid')
        move_files(test_files, 'test')
        print(f"Image-level split complete: {len(image_files) - len(val_files) - len(test_files)} train, {len(val_files)} val, {len(test_files)} test")
        remove_yolo_label_caches(base_path)
    else:
        print('Existing valid/test split detected. Keeping downloaded split.')

In [ ]:
import yaml

with open(data_yaml_path, 'r', encoding='utf-8') as handle:
    data_config = yaml.safe_load(handle)
data_config['train'] = str(DATASET_DIR / 'train' / 'images')
data_config['val'] = str(DATASET_DIR / 'valid' / 'images')
data_config['test'] = str(DATASET_DIR / 'test' / 'images')
with open(data_yaml_path, 'w', encoding='utf-8') as handle:
    yaml.safe_dump(data_config, handle, sort_keys=False)

def save_split_manifest():
    rows = []
    for split in ['train', 'valid', 'test']:
        for image_path in image_files_in_split(split):
            group_key, disease, shrimp_id, image_number = parse_shrimp_group_key(image_path.name)
            label_path = DATASET_DIR / split / 'labels' / f'{image_path.stem}.txt'
            label_lines = [line for line in label_path.read_text(encoding='utf-8').splitlines() if line.strip()] if label_path.exists() else []
            rows.append({'split': split, 'image': image_path.name, 'group_key': group_key,
                         'disease': str(disease).lower(), 'shrimp_id': shrimp_id,
                         'mask_instances': len(label_lines), 'is_labeled': bool(label_lines)})
    manifest = pd.DataFrame(rows).sort_values(['split', 'image']).reset_index(drop=True)
    manifest_path = REPORT_DIR / 'stratified_grouped_specimen_seed42_manifest.csv'
    manifest.to_csv(manifest_path, index=False)
    split_hashes = []
    for split in ['train', 'valid', 'test']:
        names = manifest.loc[manifest['split'] == split, 'image'].tolist()
        split_hashes.append(hashlib.sha256('\n'.join(names).encode()).hexdigest())
    fingerprint = hashlib.sha256('||'.join(split_hashes).encode()).hexdigest()
    (REPORT_DIR / 'stratified_grouped_specimen_seed42_fingerprint.txt').write_text(fingerprint + '\n', encoding='utf-8')
    return manifest, fingerprint

import hashlib
import pandas as pd
manifest, split_fingerprint = save_split_manifest()
test_paths = image_files_in_split('test')
print('Test images:', len(test_paths))
print('Split fingerprint:', split_fingerprint)


## Fourier Timing and Inference Timing

In [ ]:
import gc
import json
import time

import cv2
import numpy as np
import torch
from IPython.display import display
from ultralytics import YOLO

def cuda_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def fourier_highpass_batch(images, sigma=FOURIER_SIGMA, alpha=FOURIER_ALPHA):
    arr = np.stack(images, axis=0).astype(np.float32)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tensor = torch.from_numpy(arr).to(device=device).permute(0, 3, 1, 2).contiguous()
    height, width = tensor.shape[-2:]
    y = torch.arange(height, dtype=torch.float32, device=device) - height / 2.0
    x = torch.arange(width, dtype=torch.float32, device=device) - width / 2.0
    yy, xx = torch.meshgrid(y, x, indexing='ij')
    lowpass = torch.exp(-(xx * xx + yy * yy) / (2.0 * float(sigma) * float(sigma)))[None, None]
    freq = torch.fft.fftshift(torch.fft.fft2(tensor, dim=(-2, -1)), dim=(-2, -1))
    low = torch.fft.ifft2(torch.fft.ifftshift(freq * lowpass, dim=(-2, -1)), dim=(-2, -1)).real
    enhanced = torch.clamp(tensor + float(alpha) * (tensor - low), 0, 255)
    cuda_sync()
    return [item for item in enhanced.permute(0, 2, 3, 1).byte().cpu().numpy()]

def timed_fourier_transform(images):
    transformed = []
    durations = []
    for start in range(0, len(images), FOURIER_BATCH_SIZE):
        batch = images[start:start + FOURIER_BATCH_SIZE]
        cuda_sync()
        started = time.perf_counter()
        output = fourier_highpass_batch(batch)
        cuda_sync()
        durations.append(time.perf_counter() - started)
        transformed.extend(output)
    return transformed, durations

def timed_yolo_prediction(model, images):
    warmup_count = min(WARMUP_IMAGES, len(images))
    for image in images[:warmup_count]:
        model.predict(source=image, imgsz=IMAGE_SIZE, device=DEVICE, verbose=False, plots=False)
    durations = []
    for image in images[warmup_count:]:
        cuda_sync()
        started = time.perf_counter()
        model.predict(source=image, imgsz=IMAGE_SIZE, device=DEVICE, verbose=False, plots=False)
        cuda_sync()
        durations.append(time.perf_counter() - started)
    return durations, warmup_count

raw_images = []
for image_path in test_paths:
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        raise RuntimeError(f'Could not read test image: {image_path}')
    raw_images.append(image)

# Transform W1 once and reuse the exact transformed test images.
transformed_images, fourier_batch_seconds = timed_fourier_transform(raw_images)
fourier_seconds = sum(fourier_batch_seconds)
mean_fourier_ms = 1000.0 * fourier_seconds / len(raw_images) if raw_images else float('nan')

timing_rows = []
missing_paths = []
for method_key, model_path in MODEL_PATHS.items():
    model_path = Path(model_path)
    if not model_path.exists():
        missing_paths.append({'method_key': method_key, 'model_path': str(model_path)})
        print(f'SKIP missing checkpoint: {method_key} -> {model_path}')
        continue

    print('Timing:', method_key, '|', model_path)
    model = YOLO(str(model_path))
    inference_images = transformed_images if method_key == 'w1_fourier_highpass' else raw_images
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    prediction_seconds, warmup_count = timed_yolo_prediction(model, inference_images)
    prediction_ms = np.asarray(prediction_seconds, dtype=np.float64) * 1000.0
    mean_inference_ms = float(np.mean(prediction_ms)) if len(prediction_ms) else float('nan')
    row = {
        'method_key': method_key,
        'model_path': str(model_path),
        'test_images': len(raw_images),
        'timed_prediction_images': len(prediction_seconds),
        'warmup_images': warmup_count,
        'device': str(torch.device('cuda' if torch.cuda.is_available() else 'cpu')),
        'image_size': IMAGE_SIZE,
        'fourier_sigma': FOURIER_SIGMA if method_key == 'w1_fourier_highpass' else None,
        'fourier_alpha': FOURIER_ALPHA if method_key == 'w1_fourier_highpass' else None,
        'fourier_transform_measured_for_method': method_key == 'w1_fourier_highpass',
        'mean_fourier_ms_per_image': mean_fourier_ms if method_key == 'w1_fourier_highpass' else 0.0,
        'mean_yolo_inference_ms_per_image': mean_inference_ms,
        'median_yolo_inference_ms_per_image': float(np.median(prediction_ms)) if len(prediction_ms) else float('nan'),
        'p95_yolo_inference_ms_per_image': float(np.percentile(prediction_ms, 95)) if len(prediction_ms) else float('nan'),
        'effective_mean_ms_per_image': (mean_fourier_ms if method_key == 'w1_fourier_highpass' else 0.0) + mean_inference_ms,
        'peak_gpu_memory_mb': (torch.cuda.max_memory_allocated() / (1024 ** 2)) if torch.cuda.is_available() else 0.0,
        'split_fingerprint': split_fingerprint,
    }
    timing_rows.append(row)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

timing_df = pd.DataFrame(timing_rows)
timing_csv = REPORT_DIR / 'inference_timing_all_methods_seed42.csv'
timing_json = REPORT_DIR / 'inference_timing_all_methods_seed42.json'
timing_df.to_csv(timing_csv, index=False)
timing_json.write_text(json.dumps({'rows': timing_rows, 'missing_paths': missing_paths}, indent=2), encoding='utf-8')
display(timing_df)
print('Saved:', timing_csv)
print('Saved:', timing_json)
if missing_paths:
    print('Missing checkpoints were skipped:', missing_paths)

del raw_images, transformed_images
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Interpretation

Use `inference_timing_all_methods_seed42.csv` for the comparison.

For W1:

`effective_mean_ms_per_image = mean_fourier_ms_per_image + mean_yolo_inference_ms_per_image`

For the other methods, Fourier time is zero and effective latency equals YOLO inference time.
The timing excludes image disk loading and output rendering, but includes Ultralytics image
preprocessing, model forward pass, and postprocessing. Warm-up images are excluded from the
reported prediction statistics.
